# 银行业务量预测 · V2 多模型比较

**用途：接续 V1，先备份旧结果，再比较不同模型。推荐在原 Colab 环境运行全部单元。**

本 Notebook 交付时未执行模型实验，不预填优化成绩。所有文件写入独立的 `bank_outputs_v2/run_时间戳/`，不会覆盖 `/content/bank_outputs`。第一步自动把 V1 完整结果打包；最后自动生成 V2 ZIP，并在 Colab 提供下载。

### 当前版本已知结果（来自用户截图，不是本文件重新测得）
| 项目 | 数值 |
|---|---:|
| V1：A Recent-8；B 无近期加权 XGBoost | 7–9 月综合 MAPE 11.188% |
| V1 九月综合 MAPE | 14.304% |
| Python 实现的 R 固定方案 | 11.545% |
| 历史 R 最佳参考 | 11.033% |

Recent 基线已基本复现，XGBoost 尚有差异。V2 同时保留 **同环境重跑的 V1 策略** 与 **旧 CSV 的实际结果（若提供）**，避免混淆环境差异和算法改进。

### 为什么这样优化
1. **Ridge**：低方差正则化回归，用近期样本缓解早期异常影响；比较有无线性时间趋势。
2. **ExtraTrees**：更强随机化的树集成，检查小样本下另一种非线性模型是否更稳健。
3. **CatBoost**：直接处理日期类别。比较 log-RMSE 与 `MAE + 1/(actual+1)` 样本权重，后者的训练目标与官方比例误差成比例。
4. **B 水平×月内形状**：将月际水平和月内位置分开估计，历史月份内归一化后学习形状，避免把上月月末高值直接当成下月全月水平。包含无形状对照与受限漂移消融。
5. **固定 50% 融合**：各新模型与 V1 浮点预测融合，仅在开发期比较，不搜索大量融合权重。

### 验证规则
7–8 月为开发窗口；另报告 5–6 月稳健性诊断，但不纳入选择分数。先冻结配置、模型与代码哈希，再计算 9 月结果。**9 月已被前期实验和截图观察，本轮只能称“事后时间检验”，不能称全新独立测试。** 新模型设计已经受既有诊断启发，不能消除这一限制。

最终选择规则预先固定：新模型在 7–8 月整体至少改善 0.10 个百分点，且任一开发月不比同环境 V1 差超过 1.0 个百分点；否则保留 V1。这些阈值是保守工程选择，不代表统计显著性。禁止依据 9 月排名修改最终选择。


## 0. 配置、备份旧结果与运行目录
默认 Colab 原始文件在 `/content`。如果只保留了 ZIP，上传 `营销分析--HW02.zip` 即可；也可上传训练、测试、日历三个 Excel。旧结果 ZIP 可放入 `V1_ZIP` 指定路径，使用单独目录读取。无需上传无关文件。

下载的 ZIP 包含运行结果、预测与完整实验记录；请另外保存执行后的 Notebook，以保留单元输出。


In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util, shutil, zipfile, json, hashlib, platform, warnings, inspect
from datetime import datetime
try:
    from google.colab import files as colab_files
    IN_COLAB=True
except ImportError:
    IN_COLAB=False
ROOT=Path('/content') if Path('/content').is_dir() else Path.cwd()
V1_DIR=ROOT/'bank_outputs'
V1_ZIP=None  # 如旧目录已丢失，填写旧结果ZIP的完整路径
RAW_ZIP=ROOT/'营销分析--HW02.zip'
DATA=ROOT/'bank_data'
STAMP=datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT=ROOT/'bank_outputs_v2'/('run_'+STAMP)
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'figures').mkdir()
DATA.mkdir(exist_ok=True)
SEED=2026
INSTALL_MISSING=True
USE_CATBOOST=True
AUTO_DOWNLOAD=True
DEV_STARTS=['2020-07-01','2020-08-01']
EARLY_STARTS=['2020-05-01','2020-06-01']
REQUIRED_GAIN_PP=.10
MAX_MONTH_REGRESSION_PP=1.0

def safe_extract_selected(archive,dest,allowed=None):
    dest=Path(dest).resolve()
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            if item.is_dir(): continue
            rel=Path(item.filename)
            if rel.is_absolute() or '..' in rel.parts: raise ValueError('ZIP包含不安全路径')
            if allowed is not None:
                if rel.name not in allowed: continue
                target=dest/rel.name
            else: target=dest/rel
            if not target.resolve().is_relative_to(dest): raise ValueError('ZIP越界')
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(z.read(item))

if V1_ZIP:
    restored=OUT/'v1_restored'
    safe_extract_selected(V1_ZIP,restored)
    candidates=list(restored.rglob('all_oof_predictions.csv'))
    if len(candidates)!=1: raise ValueError('旧ZIP需恰好有一份all_oof_predictions.csv')
    V1_DIR=candidates[0].parent

v1_archive=None
if V1_DIR.is_dir():
    v1_archive=shutil.make_archive(str(ROOT/('bank_V1_backup_'+STAMP)), 'zip',
                                 root_dir=V1_DIR.parent,base_dir=V1_DIR.name)
    print('V1完整备份：',v1_archive)
else:
    print('未发现V1实际运行目录；将重跑固定V1策略，不虚构旧CSV。')
print('V2输出目录：',OUT)


In [ ]:
dependencies={'numpy':'numpy','pandas':'pandas','openpyxl':'openpyxl',
              'matplotlib':'matplotlib','sklearn':'scikit-learn','xgboost':'xgboost'}
if USE_CATBOOST: dependencies['catboost']='catboost'
missing=[package for module,package in dependencies.items() if importlib.util.find_spec(module) is None]
if missing:
    if not INSTALL_MISSING: raise ImportError('请安装：'+', '.join(missing))
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from IPython.display import display, FileLink
if USE_CATBOOST:
    from catboost import CatBoostRegressor
np.random.seed(SEED)
versions={p:importlib.metadata.version(p) for p in dependencies.values()}
(OUT/'requirements_resolved.txt').write_text('\n'.join(k+'=='+v for k,v in versions.items()),encoding='utf-8')
needed={'train_v1-202001-202009.xlsx','test_v1.xlsx','wkd_v1.xlsx'}
if RAW_ZIP.exists(): safe_extract_selected(RAW_ZIP,DATA,needed)
for name in needed:
    if not (DATA/name).exists() and (ROOT/name).exists(): shutil.copy2(ROOT/name,DATA/name)
if any(not (DATA/name).exists() for name in needed):
    if IN_COLAB:
        from google.colab import files
        print('请上传原始数据ZIP，或三个Excel文件。')
        uploaded=files.upload()
        for name in uploaded:
            path=Path(name)
            if path.suffix.lower()=='.zip': safe_extract_selected(path,DATA,needed)
            elif path.name in needed: shutil.copy2(path,DATA/path.name)
for name in needed: assert (DATA/name).exists(), f'缺少 {name}，请检查DATA与RAW_ZIP'
manifest={'python':platform.python_version(),'versions':versions,'seed':SEED,
          'input_sha256':{name:hashlib.sha256((DATA/name).read_bytes()).hexdigest() for name in sorted(needed)}}
(OUT/'environment.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
display(versions)


## 1. 沿用 V1 数据审计与已知日历特征
不改聚合、评分样本或取整规则。输入断言失败时停止，不自动删除、补零或修补记录。所有岗位保持同一组完整月验证日期。日历来自已知的工作日安排，不使用未来业务量。


In [ ]:
def parse_dates(series):
    def one(v):
        if pd.isna(v):
            return pd.NaT
        if isinstance(v, (int,float,np.integer,np.floating)):
            return pd.Timestamp('1899-12-30') + pd.to_timedelta(float(v),unit='D')
        s = str(v).strip()
        if s.isdigit() and len(s)==8:
            return pd.to_datetime(s,format='%Y%m%d',errors='raise')
        if s.replace('.','',1).isdigit():
            return pd.Timestamp('1899-12-30') + pd.to_timedelta(float(s),unit='D')
        return pd.to_datetime(v,errors='raise')
    return series.map(one).pipe(pd.to_datetime).dt.normalize()

raw = pd.read_excel(DATA / 'train_v1-202001-202009.xlsx')
test_raw = pd.read_excel(DATA / 'test_v1.xlsx')
wkd = pd.read_excel(DATA / 'wkd_v1.xlsx')
assert {'date','post_id','biz_type','periods','amount'} <= set(raw)
assert {'date','post_id'} <= set(test_raw)
assert {'ORIG_DT','WKD_TYP_CD'} <= set(wkd)
raw['date'] = parse_dates(raw.date)
test_raw['date'] = parse_dates(test_raw.date)
for frame in [raw,test_raw]:
    frame['post_id'] = frame.post_id.astype('string').str.strip().str.upper()
raw['biz_type'] = raw.biz_type.astype('string').str.strip().str.upper()
raw['amount'] = pd.to_numeric(raw.amount,errors='raise')
raw['periods'] = pd.to_numeric(raw.periods,errors='raise')
cal = pd.DataFrame({'date':parse_dates(wkd.ORIG_DT),
                   'wkd_typ_cd':wkd.WKD_TYP_CD.astype('string').str.strip().str.upper()})
audit = pd.DataFrame({'missing':raw.isna().sum(),'dtype':raw.dtypes.astype(str)})
display(audit)
audit.to_csv(OUT/'data_audit.csv',encoding='utf-8-sig')
assert not raw[['date','post_id','biz_type','periods','amount']].isna().any().any()
assert not test_raw[['date','post_id']].isna().any().any()
assert not cal.isna().any().any()
assert np.isfinite(raw.amount).all() and raw.amount.ge(0).all()
assert raw.periods.isin(range(1,49)).all()
assert set(raw.post_id)=={'A','B'} and set(test_raw.post_id)=={'A','B'}
assert set(raw.loc[raw.post_id=='A','biz_type'])=={f'A{i}' for i in range(1,14)}
assert set(raw.loc[raw.post_id=='B','biz_type'])=={'B1'}
keys=['date','post_id','biz_type','periods']
dupes=raw[raw.duplicated(keys,keep=False)]
dupes.to_csv(OUT/'duplicate_keys.csv',index=False,encoding='utf-8-sig')
assert dupes.empty, '有重复原始主键，请检查 duplicate_keys.csv；未自动去重。'
assert not cal.date.duplicated().any()
assert cal.wkd_typ_cd.isin(['WN','WS','SN','SS','NH']).all()
assert not test_raw.duplicated(['date','post_id']).any()
coverage=raw.groupby(['date','post_id','biz_type']).agg(n_periods=('periods','nunique'),n_rows=('amount','size'))
coverage.to_csv(OUT/'period_coverage.csv',encoding='utf-8-sig')
display(coverage.groupby('post_id').agg(['min','max']))
if not coverage.n_periods.eq(48).all():
    warnings.warn('并非每个日期/业务都有48个时段，请核实是否采用稀疏零值存储；不补造交易。')
daily_base=raw.groupby(['date','post_id'],as_index=False).amount.sum()
expected=pd.MultiIndex.from_product([pd.date_range('2020-01-01','2020-09-30'),['A','B']],names=['date','post_id'])
assert daily_base.set_index(['date','post_id']).index.sort_values().equals(expected.sort_values())
test_keys=pd.MultiIndex.from_frame(test_raw[['date','post_id']])
expected_test=pd.MultiIndex.from_product([pd.date_range('2020-10-01','2020-10-31'),['A','B']],names=['date','post_id'])
assert len(test_raw)==62 and test_keys.sort_values().equals(expected_test.sort_values())
assert np.isclose(daily_base.amount.sum(),raw.amount.sum())
print('日度记录：',len(daily_base),'；测试记录：',len(test_raw))


In [ ]:
cal=cal.sort_values('date').reset_index(drop=True)
assert cal.date.diff().dropna().eq(pd.Timedelta(days=1)).all(), '日历不连续'
assert cal.date.min()<=pd.Timestamp('2020-01-01') and cal.date.max()>=pd.Timestamp('2020-10-31')
cal['yr']=cal.date.dt.year
cal['month']=cal.date.dt.month
cal['dom']=cal.date.dt.day
cal['dow']=cal.date.dt.dayofweek+1
cal['day_of_year']=cal.date.dt.dayofyear
cal['time_index']=(cal.date-cal.date.min()).dt.days
cal['days_to_month_end']=cal.date.dt.days_in_month-cal.dom
cal['is_work']=cal.wkd_typ_cd.isin(['WN','WS']).astype(int)
for typ in ['WN','WS','SN','SS','NH']:
    cal['type_'+typ]=cal.wkd_typ_cd.eq(typ).astype(int)
g=cal.groupby(['yr','month'])['is_work']
cal['workday_number']=g.cumsum()
cal['workdays_left']=g.transform('sum')-cal.workday_number
cal['prev_break']=0
cal['next_break']=0
work=cal.is_work.to_numpy()
for i in range(len(cal)):
    if work[i]:
        j=i-1
        while j>=0 and work[j]==0:
            cal.loc[i,'prev_break']+=1
            j-=1
        j=i+1
        while j<len(cal) and work[j]==0:
            cal.loc[i,'next_break']+=1
            j+=1
cal['work_fraction']=cal.workday_number/(cal.workday_number+cal.workdays_left).clip(lower=1)
cal['quarter_end']=cal.month.isin([3,6,9,12]).astype(int)
cal['quarter_position']=cal.quarter_end*cal.work_fraction
cal['first_after_break']=cal.prev_break.ge(3).astype(int)
cal['last_before_break']=cal.next_break.ge(3).astype(int)
nh=cal.loc[cal.wkd_typ_cd.eq('NH'),'date']
cal['days_since_NH']=[min([30]+[(d-h).days for h in nh if h<=d]) for d in cal.date]
cal['days_until_NH']=[min([30]+[(h-d).days for h in nh if h>=d]) for d in cal.date]
R_FEATURES=['month','dom','dow','day_of_year','time_index','days_to_month_end',
            'workday_number','workdays_left','prev_break','next_break','is_work',
            'type_WN','type_WS','type_SN','type_SS','type_NH']
EXTRA_FEATURES=['work_fraction','quarter_end','quarter_position','first_after_break',
                'last_before_break','days_since_NH','days_until_NH']
daily=daily_base.merge(cal,on='date',how='left',validate='many_to_one').sort_values(['post_id','date']).reset_index(drop=True)
test=test_raw[['date','post_id']].assign(row_id=np.arange(len(test_raw))).merge(cal,on='date',how='left',validate='many_to_one').sort_values('row_id').reset_index(drop=True)
assert not daily[R_FEATURES].isna().any().any() and not test[R_FEATURES].isna().any().any()
daily.to_csv(OUT/'daily_aggregated.csv',index=False,encoding='utf-8-sig')
display(daily.groupby(['post_id','wkd_typ_cd']).amount.agg(['size','mean','min','max']))
display(daily.query("post_id=='B' and is_work==0").amount.describe())


## 2. 沿用评分与 V1 模型函数
官方指标为 `100 × mean(abs(y-p)/(y+1))`；B 仅工作日；两个岗位等权平均。先融合浮点值，再统一取整。V1 固定策略为 A Recent-8、B 无近期样本加权 XGBoost，沿用上次截图选出的模型，**不重新搜索旧方案**。


In [ ]:
def postprocess(pred,future):
    p=np.maximum(0,np.rint(np.asarray(pred,dtype=float)))
    p[(future.post_id.to_numpy()=='B') & (future.is_work.to_numpy()==0)]=0
    assert np.isfinite(p).all()
    return p.astype(np.int64)

def metrics(frame):
    z=frame[(frame.post_id=='A') | frame.is_work.eq(1)]
    assert len(z)>0 and z.amount.notna().all() and z.pred.notna().all()
    e=z.pred-z.amount
    return {'n':len(z),'MAPE':float((e.abs()/(z.amount+1)).mean()*100),
            'Bias':float(e.mean()),'MAE':float(e.abs().mean()),
            'Actual_Mean':float(z.amount.mean()),'Predict_Mean':float(z.pred.mean())}

def score_table(frame,groups):
    records=[]
    for key,h in frame.groupby(groups,sort=False):
        if not isinstance(key,tuple): key=(key,)
        records.append(dict(zip(groups,key)) | metrics(h))
    return pd.DataFrame(records)

def competition_score(frame):
    scores=score_table(frame,['post_id']).set_index('post_id')
    assert {'A','B'}<=set(scores.index)
    return float(scores.loc[['A','B'],'MAPE'].mean())

def recent(history,future,k=8,b_mode=False):
    h=history.sort_values('date')
    if h.post_id.iloc[0]=='B': h=h[h.is_work==1]
    result=[]
    for row in future.itertuples():
        if b_mode and row.is_work==0:
            result.append(0.); continue
        pool=h[h.wkd_typ_cd==row.wkd_typ_cd]
        if b_mode:
            pool=pool[pool.dow==row.dow]
            if len(pool)<2: pool=h[h.dow==row.dow]
        elif row.wkd_typ_cd in ['WN','SN']:
            same=pool[pool.dow==row.dow]
            if len(same)>=3: pool=same
        if pool.empty: pool=h
        pool=pool.tail(k)
        result.append(np.average(pool.amount,weights=np.arange(1,len(pool)+1)))
    return np.array(result)

def level_baseline(history,future,window=20):
    h=history[history.is_work==1].sort_values('date')
    r=h.tail(window); season=h.tail(80)
    level=np.average(r.amount,weights=np.arange(1,len(r)+1))
    overall=season.amount.mean()
    factor=(season.groupby('dow').amount.mean()/overall).clip(.7,1.3) if overall>0 else pd.Series(dtype=float)
    return np.where(future.is_work==1,level*future.dow.map(factor).fillna(1).to_numpy(),0)

def tree_predict(history,future,kind='xgb_r',enhanced=False,tau=90,weighted=True):
    h=history.copy()
    if h.post_id.iloc[0]=='B': h=h[h.is_work==1]
    features=R_FEATURES+(EXTRA_FEATURES if enhanced else [])
    X=h[features].to_numpy(dtype=float); F=future[features].to_numpy(dtype=float)
    y=np.log1p(h.amount.to_numpy())
    age=(h.date.max()-h.date).dt.days.to_numpy()
    weights=.3+.7*np.exp(-age/tau) if weighted else np.ones(len(h))
    if kind=='hgb':
        model=HistGradientBoostingRegressor(max_iter=180,max_leaf_nodes=7,
            learning_rate=.04,l2_regularization=3,min_samples_leaf=15,
            early_stopping=False,random_state=SEED)
        model.fit(X,y,sample_weight=weights)
        pred=model.predict(F)
    else:
        params=dict(objective='reg:squarederror',max_depth=3,eta=.04,
            min_child_weight=4,subsample=.9,colsample_bytree=.9,
            reg_lambda=3,nthread=2,seed=SEED,tree_method='hist',base_score=.5)
        model=xgb.train(params,xgb.DMatrix(X,label=y,weight=weights),num_boost_round=220)
        pred=model.predict(xgb.DMatrix(F))
    return np.maximum(0,np.expm1(pred))


## 3. 新模型：小规模、可解释、有配对对照
Ridge 在每个训练折内独立拟合标准化器；类别编码是预先定义的日历类别。所有目标相关形状、水平、权重均从当前折历史计算。没有验证期 early stopping，没有随机划分。

CatBoost 的内置 MAPE 分母为 `max(1, abs(y))`，**不是比赛的 `y+1`**，本文件不使用内置 MAPE 代替官方指标。原尺度 MAE 搭配归一化 `1/(y+1)` 权重，其经验训练损失与官方未取整误差成比例；零业务量样本因此权重大，这是官方指标的真实性质，可能影响泛化，不默默裁剪权重。

官方参数说明：[CatBoost 损失函数](https://catboost.ai/docs/en/concepts/loss-functions-regression)、[CatBoost fit](https://catboost.ai/docs/en/concepts/python-reference_catboostregressor_fit)、[ExtraTrees](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesRegressor.html)。


In [ ]:
def keep_history(history,lookback=None):
    h=history.sort_values('date').copy()
    if h.post_id.iloc[0]=='B': h=h[h.is_work.eq(1)].copy()
    if lookback is not None: h=h[h.date>=history.date.max()-pd.Timedelta(days=lookback-1)].copy()
    assert len(h)>=20
    return h

def calendar_matrix(frame,origin,trend=False):
    # 固定类别，不从未来样本学习编码或分布。
    z=pd.DataFrame(index=frame.index)
    for k in ['WN','WS','SN','SS','NH']: z['type_'+k]=(frame.wkd_typ_cd==k).astype(float)
    for k in range(1,8): z['dow_'+str(k)]=(frame.dow==k).astype(float)
    for k in ['work_fraction','days_to_month_end','prev_break','next_break','is_work']:
        z[k]=frame[k].astype(float)
    z['position_sq']=z.work_fraction**2
    z['position_work']=z.work_fraction*z.is_work
    if trend: z['linear_time']=(frame.date-origin).dt.days/30
    return z.to_numpy(dtype=float)

def ridge_predict(history,future,alpha=20,trend=False,lookback=120):
    h=keep_history(history,lookback)
    X=calendar_matrix(h,h.date.min(),trend); F=calendar_matrix(future,h.date.min(),trend)
    scaler=StandardScaler().fit(X)
    model=Ridge(alpha=alpha)
    model.fit(scaler.transform(X),np.log1p(h.amount))
    return np.maximum(0,np.expm1(model.predict(scaler.transform(F))))

def extra_predict(history,future,leaf=5):
    h=keep_history(history,120)
    features=R_FEATURES+EXTRA_FEATURES
    model=ExtraTreesRegressor(n_estimators=300,max_depth=5,min_samples_leaf=leaf,
            max_features=.85,random_state=SEED,n_jobs=2)
    model.fit(h[features],np.log1p(h.amount))
    return np.maximum(0,np.expm1(model.predict(future[features])))

def cat_predict(history,future,objective='log_rmse'):
    h=keep_history(history,120)
    numeric=R_FEATURES+EXTRA_FEATURES
    X=h[numeric].copy(); F=future[numeric].copy()
    X['calendar_type']=h.wkd_typ_cd.astype(str); F['calendar_type']=future.wkd_typ_cd.astype(str)
    X['weekday_category']=h.dow.astype(str); F['weekday_category']=future.dow.astype(str)
    kwargs=dict(iterations=350,depth=3,learning_rate=.035,l2_leaf_reg=8,
                random_seed=SEED,thread_count=2,verbose=False,allow_writing_files=False)
    if objective=='official_mae':
        y=h.amount.to_numpy(dtype=float)
        w=1/(y+1); w=w/w.mean()
        model=CatBoostRegressor(loss_function='MAE',**kwargs)
    else:
        y=np.log1p(h.amount.to_numpy(dtype=float)); w=np.ones(len(h))
        model=CatBoostRegressor(loss_function='RMSE',**kwargs)
    model.fit(X,y,cat_features=['calendar_type','weekday_category'],sample_weight=w)
    p=model.predict(F)
    return np.maximum(0,p if objective=='official_mae' else np.expm1(p))

def b_level_shape(history,future,shape=True,drift=False):
    # 月内形状和月际水平分开估计：只使用当前折之前完整月份。
    h=keep_history(history)
    h=h[h.date>=pd.Timestamp('2020-03-01')].copy()
    h['ym']=h.date.dt.to_period('M')
    monthly=h.groupby('ym').amount.mean()
    assert len(monthly)>=2
    level=float(monthly.iloc[-1])
    if drift and monthly.iloc[-2]>0:
        level*=np.clip(1+.5*(monthly.iloc[-1]/monthly.iloc[-2]-1),.9,1.1)
    pred=np.zeros(len(future),dtype=float)
    fw=future[future.is_work.eq(1)].copy()
    if fw.empty: return pred
    factors=np.ones(len(fw))
    if shape:
        h['month_level']=h.ym.map(monthly)
        h=h[h.month_level.gt(0)].copy()
        def shape_matrix(f):
            pos=f.work_fraction.to_numpy()
            return np.column_stack([pos,pos**2,f.type_WS.to_numpy(),
                *[(f.dow==i).astype(float).to_numpy() for i in range(1,8)]])
        X=shape_matrix(h); F=shape_matrix(fw)
        scaler=StandardScaler().fit(X)
        model=Ridge(alpha=20)
        # log ratio的目标仅在已知历史月内计算；不用于验证月特征。
        target=np.log((h.amount.to_numpy()+1)/(h.month_level.to_numpy()+1))
        model.fit(scaler.transform(X),target)
        factors=np.clip(np.exp(model.predict(scaler.transform(F))),.5,1.8)
        # 预测月份全部工作日的已知日期特征归一化，不接触实际业务量。
        factors=factors/factors.mean()
    pred[future.is_work.to_numpy()==1]=level*factors
    return pred

def raw_predictions(history,future):
    assert 'amount' not in future and history.date.max()<future.date.min()
    post=history.post_id.iloc[0]
    v1=recent(history,future) if post=='A' else tree_predict(history,future,weighted=False)
    p={'v1_fixed':v1,'recent8':recent(history,future),
       'ridge_calendar':ridge_predict(history,future),
       'ridge_trend':ridge_predict(history,future,trend=True),
       'ridge_alpha80':ridge_predict(history,future,alpha=80),
       'extra_leaf5':extra_predict(history,future,5),
       'extra_leaf12':extra_predict(history,future,12)}
    if USE_CATBOOST:
        p['cat_log_rmse']=cat_predict(history,future)
        p['cat_official_mae']=cat_predict(history,future,'official_mae')
    if post=='B':
        p['level_only']=b_level_shape(history,future,shape=False)
        p['level_shape']=b_level_shape(history,future)
        p['level_shape_drift']=b_level_shape(history,future,drift=True)
    # 固定等权融合，不进行连续权重搜索。
    for key,value in list(p.items()):
        if key not in ['v1_fixed','recent8','level_only']:
            p[key+'_blend50']=.5*v1+.5*value
    for name,values in p.items():
        assert len(values)==len(future) and np.isfinite(values).all(),name
    return p


## 4. 固定实验清单与逐折运行
每次运行有独立目录。每完成一个折立即保存预测，训练过程中中断也不会丢失已完成折的数据。新模型均使用固定小网格，当前候选集不包含自动大规模搜索。5–6 月只用于展示跨月稳定性，避免事后改变主选择窗口。


In [ ]:
model_config={'seed':SEED,'development':DEV_STARTS,'early_diagnostics':EARLY_STARTS,
 'september_role':'post_hoc_temporal_check_not_blind',
 'ridge':{'alpha':[20,80],'lookback_days':120,'trend_variants':[False,True]},
 'extra':{'n_estimators':300,'max_depth':5,'leaf':[5,12],'lookback_days':120},
 'catboost':{'enabled':USE_CATBOOST,'iterations':350,'depth':3,'lookback_days':120,
             'objectives':['log_rmse','official_weighted_mae']},
 'shape':{'start':'2020-03-01','alpha':20,'clip':[.5,1.8],'drift_clip':[.9,1.1]},
 'selection':{'gain_pp':REQUIRED_GAIN_PP,'max_month_regression_pp':MAX_MONTH_REGRESSION_PP}}
# 在Notebook中，直接保存函数源代码的可用性取决于IPython；配置与输入哈希始终记录。
function_source={}
for fn in [ridge_predict,extra_predict,cat_predict,b_level_shape,raw_predictions]:
    try: function_source[fn.__name__]=inspect.getsource(fn)
    except (OSError,TypeError): function_source[fn.__name__]='请参见本Notebook定义单元'
(OUT/'model_config.json').write_text(json.dumps(model_config,ensure_ascii=False,indent=2),encoding='utf-8')
(OUT/'model_sources.json').write_text(json.dumps(function_source,ensure_ascii=False,indent=2),encoding='utf-8')

def run_folds(starts):
    records=[]
    for start in pd.to_datetime(starts):
        end=start+pd.offsets.MonthBegin(1)
        for post in ['A','B']:
            history=daily[(daily.post_id==post)&daily.date.lt(start)].copy()
            valid=daily[(daily.post_id==post)&daily.date.ge(start)&daily.date.lt(end)].copy()
            assert len(valid)==start.days_in_month
            print('运行',start.strftime('%Y-%m'),post,flush=True)
            bank=raw_predictions(history,valid.drop(columns='amount'))
            current=[]
            for name,pred in bank.items():
                z=valid[['date','post_id','amount','is_work','wkd_typ_cd','workday_number']].copy()
                z['fold']=start.strftime('%Y-%m');z['model']=name
                z['pred_raw']=pred;z['pred']=postprocess(pred,valid)
                current.append(z)
            fold=pd.concat(current,ignore_index=True)
            fold.to_csv(OUT/f'fold_{start:%Y%m}_{post}.csv',index=False,encoding='utf-8-sig')
            records.append(fold)
    return pd.concat(records,ignore_index=True)

dev=run_folds(DEV_STARTS)
dev.to_csv(OUT/'oof_development_78.csv',index=False,encoding='utf-8-sig')
dev_scores=score_table(dev,['post_id','model'])
dev_monthly=score_table(dev,['fold','post_id','model'])
display(dev_scores.sort_values(['post_id','MAPE']))


## 5. 只根据开发期冻结最终方案
同时保存落选原因。“未达最小提升”或“单月恶化超限”的模型仍保留完整回测记录，不从报告中删除。


In [ ]:
selected={};decisions=[]
for post in ['A','B']:
    scores=dev_scores[dev_scores.post_id.eq(post)].set_index('model')
    ref=float(scores.loc['v1_fixed','MAPE'])
    months=dev_monthly[dev_monthly.post_id.eq(post)].pivot(index='fold',columns='model',values='MAPE')
    eligible=[]
    for name,row in scores.iterrows():
        gain=ref-float(row.MAPE)
        worst_delta=float((months[name]-months['v1_fixed']).max())
        accepted=name=='v1_fixed' or (gain>=REQUIRED_GAIN_PP and worst_delta<=MAX_MONTH_REGRESSION_PP)
        reason='固定对照' if name=='v1_fixed' else ('通过' if accepted else '提升不足或单月恶化超限')
        decisions.append(dict(post_id=post,model=name,MAPE_78=row.MAPE,gain_pp=gain,
                              worst_month_delta_pp=worst_delta,eligible=accepted,reason=reason))
        if accepted: eligible.append((float(row.MAPE),name))
    selected[post]=min(eligible)[1]
decision_table=pd.DataFrame(decisions)
decision_table.to_csv(OUT/'selection_decisions.csv',index=False,encoding='utf-8-sig')
frozen={'selected':selected,'config':model_config,'input_sha256':manifest['input_sha256'],
        'model_sources_sha256':hashlib.sha256((OUT/'model_sources.json').read_bytes()).hexdigest(),
        'note':'仅7–8月选择；9月为已被观察的事后时间检验'}
(OUT/'frozen_selection.json').write_text(json.dumps(frozen,ensure_ascii=False,indent=2),encoding='utf-8')
display(decision_table.sort_values(['post_id','MAPE_78']))
print('冻结方案：',selected)


## 6. 早期稳定性与九月事后检验
以下单元不会更新 `selected`。所有候选的九月分数是诊断用途，不是新的选择入口。


In [ ]:
selected=json.loads((OUT/'frozen_selection.json').read_text(encoding='utf-8'))['selected']
early=run_folds(EARLY_STARTS)
september=run_folds(['2020-09-01'])
all_oof=pd.concat([early,dev,september],ignore_index=True)
all_oof.to_csv(OUT/'all_oof_predictions.csv',index=False,encoding='utf-8-sig')
monthly_scores=score_table(all_oof,['fold','post_id','model'])
monthly_scores.to_csv(OUT/'monthly_all_models.csv',index=False,encoding='utf-8-sig')
cv=pd.concat([dev,september],ignore_index=True)
overall_models=score_table(cv,['post_id','model'])
overall_models.to_csv(OUT/'scores_789.csv',index=False,encoding='utf-8-sig')
chosen=cv[cv.model.eq(cv.post_id.map(selected))].copy()
baseline=cv[cv.model.eq('v1_fixed')].copy()
chosen.to_csv(OUT/'selected_oof_predictions.csv',index=False,encoding='utf-8-sig')
chosen_monthly=score_table(chosen,['fold','post_id'])
chosen_monthly.to_csv(OUT/'selected_monthly_metrics.csv',index=False,encoding='utf-8-sig')
combined=chosen_monthly.pivot(index='fold',columns='post_id',values='MAPE')
combined['综合MAPE']=(combined.A+combined.B)/2
combined.to_csv(OUT/'selected_monthly_combined.csv',encoding='utf-8-sig')
overall=competition_score(chosen); base_score=competition_score(baseline)
sep_score=competition_score(chosen[chosen.fold.eq('2020-09')])
comparison=pd.DataFrame([
    ['V2冻结方案（本次运行）',overall,sep_score],
    ['V1固定策略（同环境重跑）',base_score,competition_score(baseline[baseline.fold.eq('2020-09')])],
    ['V1用户截图（四舍五入）',11.188,14.304],
    ['R历史参考（不同实现）',11.033,np.nan]],columns=['source','MAPE_789','MAPE_9'])
comparison.to_csv(OUT/'version_comparison.csv',index=False,encoding='utf-8-sig')
display(comparison);display(chosen_monthly);display(combined)
# 配对消融：有趋势/无趋势、正则强度、树叶大小、目标函数、水平/形状/漂移。
pairs=[('ridge_trend','ridge_calendar'),('ridge_alpha80','ridge_calendar'),
       ('extra_leaf12','extra_leaf5'),('cat_official_mae','cat_log_rmse'),
       ('level_shape','level_only'),('level_shape_drift','level_shape')]
paired=[]
for changed,base in pairs:
    left=monthly_scores[monthly_scores.model.eq(changed)]
    right=monthly_scores[monthly_scores.model.eq(base)][['fold','post_id','MAPE']].rename(columns={'MAPE':'base_MAPE'})
    z=left.merge(right,on=['fold','post_id'],validate='one_to_one')
    if not z.empty:
        z['reference_model']=base;z['delta_pp']=z.MAPE-z.base_MAPE;paired.append(z)
ablation=pd.concat(paired,ignore_index=True)
ablation.to_csv(OUT/'ablation_paired.csv',index=False,encoding='utf-8-sig')
display(ablation[['fold','post_id','model','reference_model','delta_pp']])


## 7. 与 V1 的真实 CSV 核对
优先读取旧 `frozen_selection.json`，不假定每次 V1 都选中相同模型。对齐日期、岗位及真实值后，才计算跨版本差异。找不到旧文件时明确标记，绝不把截图数字扩展成伪造的逐日预测。


In [ ]:
old_file=V1_DIR/'all_oof_predictions.csv'
old_comparison=None
if old_file.exists():
    old=pd.read_csv(old_file,parse_dates=['date'])
    selection_file=V1_DIR/'frozen_selection.json'
    if not selection_file.exists(): raise FileNotFoundError('V1缺少冻结模型记录，不能可靠定位旧模型预测')
    old_selected=json.loads(selection_file.read_text(encoding='utf-8'))['selected']
    old=old[old.model.eq(old.post_id.map(old_selected))].copy()
    assert not old.duplicated(['date','post_id']).any()
    merged=chosen.merge(old[['date','post_id','amount','pred']],on=['date','post_id'],
                        how='outer',suffixes=('_v2','_v1'),indicator=True,validate='one_to_one')
    assert merged['_merge'].eq('both').all(), 'V1与V2验证日期不一致'
    assert np.allclose(merged.amount_v2,merged.amount_v1), 'V1与V2真实量不一致'
    merged['prediction_delta']=merged.pred_v2-merged.pred_v1
    merged.to_csv(OUT/'V1_V2_daily_comparison.csv',index=False,encoding='utf-8-sig')
    old_comparison=competition_score(old)
    print(f'读取V1实际CSV综合MAPE：{old_comparison:.6f}%')
    print(f'V2相对旧CSV差值：{overall-old_comparison:+.6f}个百分点（负值更好）')
    old_env=V1_DIR/'environment.json'
    if old_env.exists():
        previous=json.loads(old_env.read_text(encoding='utf-8'))
        display(pd.DataFrame({'V1':previous.get('versions',{}),'V2':versions}))
else:
    print('未找到V1逐日结果：仅提供同环境固定策略重跑和截图级参考。')


## 8. 可视化、完整历史重训与十月输出
最终模型不因九月表现而改变。十月没有真实值，不计算十月 MAPE。完整候选的十月预测也保留供诊断，不能据“看起来合理”挑选输出。


In [ ]:
fig,axes=plt.subplots(2,1,figsize=(12,7),constrained_layout=True)
for ax,post in zip(axes,['A','B']):
    h=chosen[chosen.post_id.eq(post)]
    if post=='B': h=h[h.is_work.eq(1)]
    ax.plot(h.date,h.amount,label='Actual',color='#222222')
    ax.plot(h.date,h.pred,label='V2 selected',color='#b45309')
    b=baseline[(baseline.post_id==post)&baseline.date.isin(h.date)]
    ax.plot(b.date,b.pred,label='V1 rerun',color='#2563ab',alpha=.7)
    ax.set_title(post);ax.legend()
fig.savefig(OUT/'figures'/'validation_comparison.png',dpi=160)
plt.show()
fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
for ax,post in zip(axes,['A','B']):
    d=monthly_scores[(monthly_scores.post_id==post)&monthly_scores.model.isin(['v1_fixed',selected[post]])]
    for name,h in d.groupby('model'): ax.plot(h.fold,h.MAPE,marker='o',label=name)
    ax.set_title(post+' monthly MAPE');ax.legend();ax.tick_params(axis='x',rotation=30)
fig.savefig(OUT/'figures'/'monthly_mape.png',dpi=160)
plt.show()

final=np.full(len(test),np.nan); future_all=[]
for post in ['A','B']:
    h=daily[daily.post_id.eq(post)].copy()
    idx=test.post_id.eq(post)
    f=test[idx].copy()
    bank=raw_predictions(h,f)
    final[idx.to_numpy()]=postprocess(bank[selected[post]],f)
    for name,pred in bank.items():
        z=f[['date','post_id','row_id','wkd_typ_cd']].copy()
        z['model']=name;z['Predict_amount']=postprocess(pred,f);future_all.append(z)
result=test[['date','post_id']].copy()
assert np.isfinite(final).all() and (final>=0).all()
result['Predict_amount']=final.astype(np.int64)
assert len(result)==62 and list(result)==['date','post_id','Predict_amount']
pd.testing.assert_frame_equal(result[['date','post_id']],test_raw[['date','post_id']].reset_index(drop=True))
assert result.loc[test.post_id.eq('B')&test.is_work.eq(0),'Predict_amount'].eq(0).all()
result.to_excel(OUT/'prediction_202010_python_v2.xlsx',index=False)
result.to_csv(OUT/'prediction_202010_python_v2.csv',index=False,encoding='utf-8-sig',date_format='%Y-%m-%d')
pd.concat(future_all).to_csv(OUT/'October_all_candidates.csv',index=False,encoding='utf-8-sig')
display(result.head(16))


## 9. 中文报告、压缩包与 Colab 下载
报告使用实际运行结果，不预设“优化成功”。输出目录包括：模型配置、版本、输入哈希、所有候选逐折预测、消融表、V1/V2 对照、最终提交、图表与说明。


In [ ]:
report=f"""# 银行业务量预测 V2 实验报告

## 冻结方案与真实结果
A：{selected['A']}；B：{selected['B']}。
7–9月综合MAPE：{overall:.3f}%。九月事后时间检验：{sep_score:.3f}%。
同环境重跑V1策略：{base_score:.3f}%。V2减V1差值：{overall-base_score:+.3f}个百分点。
旧CSV实际综合分数：{old_comparison if old_comparison is not None else '未取得旧CSV'}。
截图中的V1为11.188%，历史R参考11.033%；实现、环境与选择过程可能不同。

## 分月表现
{chosen_monthly.to_string(index=False)}

## 优化解释
只在7–8月选择，要求至少改善0.10个百分点且单月恶化不超过1.0个百分点。
九月已经被前期分析观察，因此本次检验不是独立盲测；不能据此宣称泛化必然提升。
若新方案未通过预设门槛，保留V1固定策略。没有依据九月更换方案。
模型包括Ridge、ExtraTrees、{'CatBoost、' if USE_CATBOOST else ''}B水平与月内形状，以及固定50%融合。
不同月份的消融差异见ablation_paired.csv，负值表示改动降低MAPE。
5–6月用于稳定性诊断，不纳入主选择分数。单月改善不能证明长期稳健。
CatBoost加权MAE在训练集上与未取整官方误差成比例，但不保证样本外更好。
所有形状目标和水平来自折内历史；验证月整月预测，不注入验证实际值。

## 局限
样本不足一年、长假稀少、8/9月水平变化来源未知。树模型难以外推，线性趋势也可能过度外推。
B非工作日置零沿用比赛提交策略，不代表所有历史非工作日业务必定为零。
无十月真实标签，不能报告十月测试准确率。官方规则PDF仍未独立核对。

## 文件
prediction_202010_python_v2.xlsx / .csv：冻结方案的十月结果。
all_oof_predictions.csv：5–9月所有候选逐日回测。
selected_oof_predictions.csv：7–9月冻结策略结果。
selection_decisions.csv：开发期选择与落选原因。
version_comparison.csv：同环境V1、V2与历史参考。
frozen_selection.json：选模记录。environment.json：软件版本与输入哈希。
"""
(OUT/'中文技术报告.md').write_text(report,encoding='utf-8')
readme='运行结果来自本Notebook实际执行。V1目录不会覆盖；V2每次运行写独立时间戳目录。\n保存执行后的Notebook以保留全部代码和单元输出。'
(OUT/'README.txt').write_text(readme,encoding='utf-8')
checksums={str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest()
           for p in OUT.rglob('*') if p.is_file()}
(OUT/'output_checksums.json').write_text(json.dumps(checksums,ensure_ascii=False,indent=2),encoding='utf-8')
v2_zip=shutil.make_archive(str(ROOT/('bank_V2_results_'+STAMP)), 'zip',root_dir=OUT.parent,base_dir=OUT.name)
print('V1备份：',v1_archive)
print('V2结果目录：',OUT.resolve())
print('V2压缩包：',v2_zip)
display(FileLink(v2_zip))
if AUTO_DOWNLOAD and IN_COLAB:
    from google.colab import files
    files.download(v2_zip)


In [ ]:
# 如果也需要下载运行前的V1备份，单独运行本单元即可。
DOWNLOAD_V1_BACKUP=False
if DOWNLOAD_V1_BACKUP and v1_archive:
    if IN_COLAB:
        from google.colab import files
        files.download(v1_archive)
    else:
        display(FileLink(v1_archive))
